# Bloco 05 — Escrita de Dados e Projeto Final

Neste bloco vamos aprender a **gravar dados** em diferentes formatos e modos, e depois aplicar tudo
que aprendemos nos Blocos 01–04 em um **projeto final completo** com o dataset Northwind:

- Formatos de escrita (Delta, Parquet, CSV, JSON)
- Modos de escrita (`overwrite`, `append`, `error`, `ignore`)
- Particionamento na escrita (`partitionBy`)
- `repartition()` vs `coalesce()`
- **Projeto Final** — Pipeline completo: leitura → transformação → relatórios → escrita

## Setup

In [0]:
# pyspark.sql.functions — modulo principal de funcoes do PySpark.
# Contem funcoes para manipulacao de colunas, agregacoes, strings, datas, etc.
# Importamos como "F" por convencao para evitar conflito com built-ins do Python.

# pyspark.sql.window.Window — classe para definir janelas (window functions).
# Usada no Projeto Final para calcular receita YTD (acumulada) e segmentacao
# de clientes com ntile(). Sintaxe: Window.partitionBy("grupo").orderBy("ordem")

# Criar Volume para armazenar arquivos (substitui DBFS /tmp)

---
## 1. Formatos de Escrita

O Spark suporta diversos formatos de saída. Vamos ver os principais:

| Formato  | Uso típico                         | Vantagem principal             |
|----------|------------------------------------|--------------------------------|
| **Delta**   | Tabelas no Lakehouse            | ACID, time travel, otimização  |
| **Parquet** | Data lake / interchange          | Colunar, compacto, rápido      |
| **CSV**     | Exportação / integração simples  | Legível, universal             |
| **JSON**    | APIs / logs                      | Semi-estruturado, flexível     |

### 1.1 Delta (formato preferido no Databricks)

In [0]:
# Criando um DataFrame simples para demonstração

# Escrita em formato Delta como tabela gerenciada

# Verificar

### 1.2 Parquet (formato colunar)

In [0]:
# Escrita em Parquet

# Leitura de volta

### 1.3 CSV (para exportação)

In [0]:
# Escrita em CSV com cabeçalho

# Leitura de volta

### 1.4 JSON

In [0]:
# Escrita em JSON

# Leitura de volta

---
## 2. Modos de Escrita

O parâmetro `mode` controla o que acontece quando o destino já existe:

| Modo               | Comportamento                                |
|--------------------|----------------------------------------------|
| `overwrite`        | Substitui os dados existentes                |
| `append`           | Adiciona aos dados existentes                |
| `error` / `errorifexists` | Falha se já existir (padrão)          |
| `ignore`           | Ignora silenciosamente se já existir         |

In [0]:
# Modo overwrite — substitui tudo

# Modo append — adiciona registros

# Modo ignore — não faz nada se já existir

In [0]:
# Modo error — lança exceção se já existir

---
## 3. Particionamento na Escrita

O método `partitionBy()` organiza os arquivos em subdiretórios baseados nos valores de uma ou mais colunas.
Isso pode **acelerar muito** as leituras quando filtramos por essas colunas.

**Quando particionar:**
- A coluna de partição tem **cardinalidade baixa** (poucos valores distintos)
- As queries frequentemente filtram por essa coluna
- O dataset é grande o suficiente (> 1 GB)

**Quando NÃO particionar:**
- A coluna tem **alta cardinalidade** (milhares de valores) — gera arquivos pequenos demais
- O dataset é pequeno — overhead maior que o benefício
- Não há padrão claro de filtro nas queries

In [0]:
# Particionar orders por ano

# Leitura filtrando por partição (Spark faz partition pruning)

---
## 4. repartition() vs coalesce()

Antes de escrever, muitas vezes queremos controlar o **número de arquivos** gerados.

| Método          | Shuffle? | Pode aumentar? | Pode diminuir? | Quando usar                        |
|-----------------|----------|----------------|----------------|------------------------------------|  
| `repartition(n)` | Sim     | Sim            | Sim            | Redistribuir dados uniformemente    |
| `coalesce(n)`    | Não     | Não            | Sim            | Reduzir partições sem shuffle       |

**Regra prática:** use `coalesce()` para reduzir (mais eficiente), `repartition()` para aumentar ou redistribuir.

In [0]:
# repartition — redistribui com shuffle

# coalesce — reduz sem shuffle

In [0]:
# Uso prático: salvar como arquivo único para exportação

---
---
## Projeto Final — Pipeline Northwind Completo

Agora vamos construir um **pipeline completo** de **leitura → transformação → escrita**,
aplicando tudo que aprendemos nos Blocos 01–05.

O objetivo é gerar **5 relatórios analíticos** a partir do dataset Northwind e salvá-los como tabelas Delta.

### Etapa 1 — Leitura das Tabelas

### Etapa 2 — Transformação (Base Analítica)

Vamos construir uma **base analítica unificada** com joins de todas as tabelas e cálculo de receita.

In [0]:
# Base: calcular receita por item

# Join principal: orders + order_details + customers + products + categories

### Relatório 1 — Receita Mensal com YTD e Crescimento

Receita mensal, acumulado no ano (YTD) e percentual de crescimento mês a mês.

### Relatório 2 — Top 10 Produtos por Receita

### Relatório 3 — Segmentação de Clientes (Quintis)

Dividimos os clientes em 5 grupos usando `ntile(5)`: grupo 1 = maiores compradores, grupo 5 = menores.

### Relatório 4 — Clientes do UK com Receita > 1000

### Relatório 5 — Receita por Categoria × Ano (Pivot)

### Etapa 3 — Escrita dos Relatórios

Salvamos todos os relatórios como **tabelas Delta** em um schema `northwind_gold` (camada Gold do Medallion).

In [0]:
# Exportar base analítica consolidada como CSV

### Verificação Final

---
## Resumo do Bloco 05

Neste bloco aprendemos:

1. **Formatos de escrita** — Delta (preferido), Parquet, CSV, JSON
2. **Modos de escrita** — `overwrite`, `append`, `error`, `ignore`
3. **Particionamento** — `partitionBy()` para organizar arquivos por coluna
4. **Controle de partições** — `repartition()` (com shuffle) vs `coalesce()` (sem shuffle)
5. **Projeto Final** — Pipeline completo com 5 relatórios analíticos salvos como tabelas Delta

### Conceitos aplicados no Projeto Final (Blocos 01–05):
- `spark.table()` para leitura (Bloco 01)
- `withColumn()`, `select()`, `filter()` para transformações (Bloco 02)
- `join()` para cruzamento de tabelas (Bloco 03)
- `groupBy()`, `agg()`, `Window`, `pivot()`, `ntile()` para agregações (Bloco 04)
- `write.format("delta").saveAsTable()` para escrita (Bloco 05)